In [6]:
import sys
from pathlib import Path
src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path))

In [13]:
from api.events.models import EventModel
from api.db.session import engine
from sqlmodel import Session, select
from timescaledb.hyperfunctions import time_bucket
from pprint import pprint

In [10]:
with Session(engine) as session:
    query = select(EventModel).order_by(EventModel.updated_at.desc()).limit(10)
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    print(compiled_query)
    print("")
    print(str(query))
    # results = session.exec(query).all()
    # print(results)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT 10

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT :param_1


In [18]:
from sqlalchemy import func
from datetime import datetime, timedelta, timezone

with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)
    pages = ['/about', '/contact', '/pages', '/pricing']
    start = datetime.now(timezone.utc) - timedelta(days=1)
    finish = datetime.now(timezone.utc)
    pprint(bucket)
    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(
            EventModel.time >= start,
            EventModel.time <= finish,
            EventModel.page.in_(pages))
        .group_by(bucket, EventModel.page)
        .order_by(bucket.desc(), EventModel.page )
    )
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)
 

<sqlalchemy.sql.functions.Function at 0x1d6d8fef290; time_bucket>
[(datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/about', 257),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/contact', 258),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/pages', 254),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/pricing', 231)]


In [19]:
%pip install faker 

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------------------------------- -------- 1.6/2.0 MB 14.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 12.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from faker import Faker
import random
import requests
events = 20
pages = ['/about', '/contact', '/pages', '/pricing', '/home', '/blog', '/features']
fake =Faker()
session_ids = [fake.uuid4() for _ in range(20)]
path = '/api/events/'
base_url = 'http://localhost:8000'
create_endpoint = f'{base_url}{path}'
referrers = ['https://google.com', 'https://bing.com', 'https://duckduckgo.com', 'https://yahoo.com', '']

for i in range(events):
    page = random.choice(pages)
    user_agent = random.choice([fake.user_agent() for _ in range(10)])
    payload = dict(
        page=page,
        user_agent=user_agent,
        ip_address=fake.ipv4_public(),
        referrer=random.choice(referrers),
        session_id=random.choice(session_ids),
        duration=random.randint(5, 300)
    )
    response = requests.post(create_endpoint, json=payload)
    print(f"Created event for page: {page} with user_agent: {user_agent}")
    if not response.ok:
        print(f"Failed to create event: {response.status_code} - {response.text}")


Created event for page: /blog with user_agent: Mozilla/5.0 (Windows NT 11.0) AppleWebKit/536.1 (KHTML, like Gecko) Chrome/36.0.848.0 Safari/536.1
Created event for page: /pages with user_agent: Mozilla/5.0 (compatible; MSIE 6.0; Windows 95; Trident/3.1)
Created event for page: /features with user_agent: Mozilla/5.0 (iPad; CPU iPad OS 13_7 like Mac OS X) AppleWebKit/534.2 (KHTML, like Gecko) CriOS/41.0.845.0 Mobile/16Q765 Safari/534.2
Created event for page: /pricing with user_agent: Mozilla/5.0 (Windows; U; Windows NT 5.2) AppleWebKit/534.45.1 (KHTML, like Gecko) Version/4.0 Safari/534.45.1
Created event for page: /pricing with user_agent: Mozilla/5.0 (Linux; Android 4.0) AppleWebKit/533.1 (KHTML, like Gecko) Chrome/26.0.840.0 Safari/533.1
Created event for page: /blog with user_agent: Opera/8.40.(Windows NT 5.2; mn-MN) Presto/2.9.172 Version/11.00
Created event for page: /contact with user_agent: Mozilla/5.0 (Macintosh; PPC Mac OS X 10_8_9 rv:2.0; kok-IN) AppleWebKit/531.16.4 (KHTML, 